# Visualizing Alignment Scores

(Random thoughts for now to be turned into something programmatic.)

The fundamental visualization objects are
* a 2d array of scores. This might be (book x chapter(aggregate)) or (chapter x verse)
* Altair heatmap, like [Simple Heatmap — Vega-Altair 5.5.0 documentation](https://altair-viz.github.io/gallery/simple_heatmap.html)

## Score the Data

Fuller details in [Scoring Alignment Data](31ScoringAlignmentData.ipynb].

In [1]:
import altair as alt
import pandas as pd

# import with an abbreviated alias
import biblealignlib as bal
from biblealignlib import burrito, autoalign

targetlang, targetid, sourceid = ("eng", "BSB", "SBLGNT")
alsetref = burrito.AlignmentSet(targetlanguage=targetlang,
        targetid=targetid,
        sourceid=sourceid,
        langdatapath=(bal.CLEARROOT / f"alignments-{targetlang}/data"))

In [4]:
# this directory should already exist and have burrito format alignments
condition = "notebookcheck"
conditiondir = alsetref.langdatapath.parent / f"exp/{targetid}/{condition}"
print(f"Conditiondir: {conditiondir}")
sc = autoalign.Scorer(referenceset=alsetref, 
                   hypothesispath=(conditiondir / f"{sourceid}-{targetid}-eflomal.json"),
                   hypothesisaltid="eflomal")

Conditiondir: /Users/sboisen/git/Clear-Bible/alignments-eng/exp/BSB/notebookcheck

        - sourcepath: /Users/sboisen/git/Clear-Bible/Alignments/data/sources/SBLGNT.tsv
        - targetpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/targets/BSB/nt_BSB.tsv
        - alignmentpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/alignments/BSB/SBLGNT-BSB-manual.json
        - tomlpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/alignments/BSB/SBLGNT-BSB-manual.toml
        


NameError: name 'normalize_strongs' is not defined

## Collect Chapter x Verse Scores for a Book



In [ ]:
mrkscore = sc.score_group(identifier="41")
mrkscore

In [ ]:
# dict: BCID -> list[VerseScore]
mrkbc = burrito.util.groupby_bc(mrkscore.verse_scores, key=lambda vs: vs.identifier[:5])
bc = "41004"
mrkbc[bc][:5]

In [ ]:
# one chapter
source = pd.DataFrame({
    'x': [vs.identifier for vs in mrkbc[bc]],
    'y': bc,
    'z': [vs.f1 for vs in mrkbc[bc]],
})
alt.Chart(source).mark_rect().encode(
    x='x:O',
    y='y:O',
    color='z:Q',
    tooltip=['F1', 'Precsion', 'Recall']
)